# 20_kmeans_clustering

Audience: junior researcher

- Challenge path: `challenges/hard/20_kmeans_clustering`
- Source spec: [challenges/hard/20_kmeans_clustering/challenge.html](../challenge.html)
- Source implementation: [challenges/hard/20_kmeans_clustering/challenge.py](../challenge.py)


## Problem Statement

The original challenge HTML is embedded below so the notebook stays close to the repo source.

<p>
  Implement the k-means clustering algorithm for 2D points. Given arrays of x and y coordinates for data points, initial centroids, and other parameters, assign each point to the nearest centroid and update the centroids iteratively. The final centroids and labels should be stored in the output arrays.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>External libraries are not permitted</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in <code>labels</code>, <code>final_centroid_x</code>, and <code>final_centroid_y</code></li>
</ul>

<h2>Example 1:</h2>
<pre>
Input:
sample_size = 4, k = 2, max_iterations = 10
data_x = [1.0, 2.0, 8.0, 9.0]
data_y = [1.0, 2.0, 8.0, 9.0]
initial_centroid_x = [1.0, 8.0]
initial_centroid_y = [1.0, 8.0]
Output: (see reference implementation for expected output)
</pre>

<h2>Constraints</h2>
<ul>
  <li>1 ≤ sample_size ≤ 1000000</li>
  <li>1 ≤ k ≤ 1000</li>
  <li>All arrays are float32 except labels, which is int32</li>

  <li>Performance is measured with <code>k</code> = 5, <code>max_iterations</code> = 30, <code>sample_size</code> = 10,000</li>
</ul>


## Framework Coverage

This notebook collects the currently available solution artifacts for PyTorch, JAX, Triton, and MLX.

## Pytorch

Source: `challenges/hard/20_kmeans_clustering/solution/solution.pytorch.py`

In [ ]:
import torch


def solve(
    data_x: torch.Tensor,
    data_y: torch.Tensor,
    labels: torch.Tensor,
    initial_centroid_x: torch.Tensor,
    initial_centroid_y: torch.Tensor,
    final_centroid_x: torch.Tensor,
    final_centroid_y: torch.Tensor,
    sample_size: int,
    k: int,
    max_iterations: int,
):
    final_centroid_x.copy_(initial_centroid_x)
    final_centroid_y.copy_(initial_centroid_y)

    for _ in range(max_iterations):
        expanded_x = data_x.view(-1, 1) - final_centroid_x.view(1, -1)
        expanded_y = data_y.view(-1, 1) - final_centroid_y.view(1, -1)
        distances = expanded_x**2 + expanded_y**2
        labels.copy_(torch.argmin(distances, dim=1))

        for i in range(k):
            mask = labels == i
            if mask.any():
                final_centroid_x[i] = data_x[mask].mean()
                final_centroid_y[i] = data_y[mask].mean()


## Jax

Source: `challenges/hard/20_kmeans_clustering/solution/solution.jax.py`

In [ ]:
from __future__ import annotations

from typing import Any

try:
    import jax
    import jax.numpy as jnp
except Exception:  # pragma: no cover - optional dependency
    jax = None
    jnp = None

import torch


def _solve_torch(
    data_x: torch.Tensor,
    data_y: torch.Tensor,
    initial_centroid_x: torch.Tensor,
    initial_centroid_y: torch.Tensor,
    sample_size: int,
    k: int,
    max_iterations: int,
):
    labels = torch.empty(sample_size, dtype=torch.int32, device=data_x.device)
    final_centroid_x = initial_centroid_x.clone()
    final_centroid_y = initial_centroid_y.clone()

    for _ in range(max_iterations):
        expanded_x = data_x.view(-1, 1) - final_centroid_x.view(1, -1)
        expanded_y = data_y.view(-1, 1) - final_centroid_y.view(1, -1)
        distances = expanded_x**2 + expanded_y**2
        labels.copy_(torch.argmin(distances, dim=1))

        for i in range(k):
            mask = labels == i
            if mask.any():
                final_centroid_x[i] = data_x[mask].mean()
                final_centroid_y[i] = data_y[mask].mean()

    return labels, final_centroid_x, final_centroid_y


def _solve_jax(
    data_x: Any,
    data_y: Any,
    initial_centroid_x: Any,
    initial_centroid_y: Any,
    sample_size: int,
    k: int,
    max_iterations: int,
):
    labels = jnp.empty((sample_size,), dtype=jnp.int32)
    final_centroid_x = jnp.array(initial_centroid_x)
    final_centroid_y = jnp.array(initial_centroid_y)

    for _ in range(max_iterations):
        expanded_x = data_x.reshape(-1, 1) - final_centroid_x.reshape(1, -1)
        expanded_y = data_y.reshape(-1, 1) - final_centroid_y.reshape(1, -1)
        distances = expanded_x**2 + expanded_y**2
        labels = jnp.argmin(distances, axis=1).astype(jnp.int32)

        for i in range(k):
            mask = labels == i
            count = jnp.sum(mask)
            safe_count = jnp.maximum(count.astype(final_centroid_x.dtype), 1.0)
            mean_x = jnp.sum(jnp.where(mask, data_x, 0.0)) / safe_count
            mean_y = jnp.sum(jnp.where(mask, data_y, 0.0)) / safe_count
            final_centroid_x = final_centroid_x.at[i].set(jnp.where(count > 0, mean_x, final_centroid_x[i]))
            final_centroid_y = final_centroid_y.at[i].set(jnp.where(count > 0, mean_y, final_centroid_y[i]))

    return labels, final_centroid_x, final_centroid_y


if jax is not None:
    solve_jax = jax.jit(
        _solve_jax,
        static_argnames=("sample_size", "k", "max_iterations"),
    )

    def solve(
        data_x: Any,
        data_y: Any,
        initial_centroid_x: Any,
        initial_centroid_y: Any,
        sample_size: int,
        k: int,
        max_iterations: int,
        labels: Any | None = None,
        final_centroid_x: Any | None = None,
        final_centroid_y: Any | None = None,
    ):
        result = solve_jax(
            data_x,
            data_y,
            initial_centroid_x,
            initial_centroid_y,
            sample_size,
            k,
            max_iterations,
        )
        if labels is not None and hasattr(labels, "copy_"):
            labels.copy_(torch.as_tensor(result[0]))
        if final_centroid_x is not None and hasattr(final_centroid_x, "copy_"):
            final_centroid_x.copy_(torch.as_tensor(result[1]))
        if final_centroid_y is not None and hasattr(final_centroid_y, "copy_"):
            final_centroid_y.copy_(torch.as_tensor(result[2]))
            return labels, final_centroid_x, final_centroid_y
        return result
else:

    def solve(
        data_x: Any,
        data_y: Any,
        initial_centroid_x: Any,
        initial_centroid_y: Any,
        sample_size: int,
        k: int,
        max_iterations: int,
        labels: Any | None = None,
        final_centroid_x: Any | None = None,
        final_centroid_y: Any | None = None,
    ):
        result = _solve_torch(
            data_x,
            data_y,
            initial_centroid_x,
            initial_centroid_y,
            sample_size,
            k,
            max_iterations,
        )
        if labels is not None and hasattr(labels, "copy_"):
            labels.copy_(result[0])
        if final_centroid_x is not None and hasattr(final_centroid_x, "copy_"):
            final_centroid_x.copy_(result[1])
        if final_centroid_y is not None and hasattr(final_centroid_y, "copy_"):
            final_centroid_y.copy_(result[2])
            return labels, final_centroid_x, final_centroid_y
        return result


## Triton

Source: `challenges/hard/20_kmeans_clustering/solution/solution.triton.py`

In [ ]:
import torch

try:
    import triton  # noqa: F401
    import triton.language as tl  # noqa: F401
except Exception:  # pragma: no cover - optional dependency
    triton = None
    tl = None


def solve(
    data_x: torch.Tensor,
    data_y: torch.Tensor,
    labels: torch.Tensor,
    initial_centroid_x: torch.Tensor,
    initial_centroid_y: torch.Tensor,
    final_centroid_x: torch.Tensor,
    final_centroid_y: torch.Tensor,
    sample_size: int,
    k: int,
    max_iterations: int,
):
    final_centroid_x.copy_(initial_centroid_x)
    final_centroid_y.copy_(initial_centroid_y)

    for _ in range(max_iterations):
        expanded_x = data_x.view(-1, 1) - final_centroid_x.view(1, -1)
        expanded_y = data_y.view(-1, 1) - final_centroid_y.view(1, -1)
        distances = expanded_x**2 + expanded_y**2
        labels.copy_(torch.argmin(distances, dim=1))

        for i in range(k):
            mask = labels == i
            if mask.any():
                final_centroid_x[i] = data_x[mask].mean()
                final_centroid_y[i] = data_y[mask].mean()


## Mlx

No solution file is present yet.

## Verification Notes

Use `python scripts/verify_matrix_solutions.py` for the local matrix-operation verifier.
GPU-only Triton validation still depends on a remote NVIDIA environment.